# Tutorial 02: Diffusion Model

This tutorial trains a diffusion model on crystalline silicon, and uses it to generate new structures. In the training
step, crystalline frames from a molecular dynamics are taken, and each atom has a Gaussian noise added to it. The model
must then predict the noise that was added to each atom. At inference, the model receives random positions and
gradually removes the noise from them.

The training puts together four pieces:

- **Noise scheduler** (`NoiseScheduler`, set by `NoiseParameters`): decides how much noise σ is added at each time
  step, from a tiny σ<sub>min</sub>, where the atoms barely move, to a large σ<sub>max</sub>, where the structure is
  completely lost.
- **Score network** (`ScoreNetwork`, here a Cartesian EGNN): looks at a noisy structure and its noise level, and
  predicts for each atom the direction that brings it back toward a realistic position: the *score*.
- **Target and loss function**: the target is what the score network should predict, here the opposite of the noise
  that was added. The loss function, a mean squared error, measures how far the prediction is from the target.
- **Target score matching** (`TargetScoreMatchingParameters`): the frames of the molecular dynamics also come with
  their forces, which already tell where each atom wants to go. At low noise, the target uses the forces instead of
  the noise, which is a cleaner target due to thermal vibrations.

The generation puts together four other pieces:

- **Langevin generator** (`LangevinGenerator`): turns random positions into a structure, noise level after noise level,
  by moving the atoms along the score with small random kicks (Langevin dynamics).
- **Predictor-corrector sampler** (`PredictorCorrectorSamplingParameters`): how the generator handles each noise level.
  The predictor removes part of the noise and goes down to the next level; the corrector then takes a few small steps
  at that level, to thermalize the atoms before going further.
- **Excision** (`NearestNeighborsExcision`): cuts an atom and its closest neighbors out of an existing structure.
- **Constrained Langevin generator** (`ConstrainedLangevinGenerator`): the same generator, but the excised atoms stay
  fixed, and the model fills in (*repaints*) the other atoms around them.

This tutorial has two parts:

- **Part 1, training** (`examples/03_train_diffusion_model.py`): train a diffusion model on configurations of Si8.
- **Part 2, generation** (`examples/04_generate_and_repaint.py`): with a pretrained model, generate new Si8 cells from
  pure noise, and repaint Si8 cells around a few fixed atoms.

To see how a diffusion model is inserted in the active learning loop, see tutorial 3.

> **Note**
>
> This tutorial needs the LAMMPS python binding (`import lammps`): it computes in-process the Stillinger-Weber
> energies and forces, for the thermal width in Part 1 and the energies of the generated cells in Part 2.

In [ ]:
import warnings

# To suppress warnings for clearer output
warnings.simplefilter("ignore")

In [ ]:
# Define the directory where artifacts will be written. Delete it if it exists to start clean.
import shutil
from diffusion_for_multi_scale_molecular_dynamics import TOP_DIR

output_path = TOP_DIR / "tutorials" / "output" / "02-diffusion_model"

# Restart from an empty directory
if output_path.is_dir():
    shutil.rmtree(output_path)

output_path.mkdir(parents=True, exist_ok=True)

# Part 1: Training

## Data

The training data is `examples/references_files/si8_database.traj`: 2000 configurations of a cubic 8-atom cell of
diamond silicon (5.44 Å), from a molecular dynamics at 300 K, labelled with their Stillinger-Weber energy and forces.
The first 80% of the frames are used for training, and the last 20% for validation. The training frames are repeated
10 times per epoch: each time, they are noised with a new noise, so that an epoch sees each configuration at 10 noise
levels.

In [ ]:
import ase.io

training_data_path = TOP_DIR / "examples" / "references_files" / "si8_database.traj"
frames = ase.io.read(training_data_path, index=":")
print(f"{len(frames)} frames of {frames[0].get_chemical_formula()}, cell lengths {frames[0].cell.lengths()} Å")

# The first 80% of the frames for training, repeated 10 times, and the last 20% for validation.
split_index = int(0.8 * len(frames))
data_directory = output_path / "data"
data_directory.mkdir()
train_trajectory_path = data_directory / "train_conf.traj"
validation_trajectory_path = data_directory / "validation_conf.traj"
ase.io.write(train_trajectory_path, frames[:split_index] * 10)
ase.io.write(validation_trajectory_path, frames[split_index:])

## Noise schedule

The `NoiseParameters` define the noise schedule: the width σ of the noise at each of the `total_time_steps` steps, from
`sigma_min_cart` to `sigma_max_cart` (in Å), here on an exponential scale from 0.01 to 5 Å. σ<sub>max</sub> must be
large enough that a noised configuration no longer remembers the original one: 5 Å is about the size of the cell. The
same schedule must be used for sampling, since a model only knows the noise levels it was trained on.
`corrector_step_epsilon` sets the size of the corrector steps during sampling (see Part 2).

The `ASEForDiffusionDataModule` prepares what is fed to the score network. It randomly selects batches of 64 clean
configurations, picks a random noise level of the schedule for each one, and adds the corresponding random noise to
its atoms. This is done efficiently: the configurations are stored once in a compact format (`processed_data/`), and
the noised batches are prepared by 4 worker processes while the network trains, then sent to the GPU if one is
available.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.data_module.diffusion.ase_for_diffusion_data_module import (
    ASEForDiffusionDataModule, ASEForDiffusionDataModuleParameters)
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.noise_schedulers.noise_parameters import \
    NoiseParameters

noise_parameters = NoiseParameters(total_time_steps=1000,
                                   schedule_type="exponential",
                                   sigma_min_cart=1e-2,
                                   sigma_max_cart=5.0,
                                   corrector_step_epsilon=1.6e-4)

data_module_parameters = ASEForDiffusionDataModuleParameters(data_source="ase_trajectory",
                                                             elements=["Si"],
                                                             batch_size=64,
                                                             num_workers=4,
                                                             max_atom=8,
                                                             use_fixed_lattice_parameters=True,
                                                             noise_parameters=noise_parameters)
data_module = ASEForDiffusionDataModule(processed_dataset_dir=str(output_path / "processed_data"),
                                        hyper_params=data_module_parameters,
                                        train_trajectory_list=[str(train_trajectory_path)],
                                        validation_trajectory_list=[str(validation_trajectory_path)],
                                        working_cache_dir=str(output_path / "cache"))

## Score network

The score network is a Cartesian EGNN (E(n)-equivariant graph neural network): each atom is a node, connected to the
atoms closer than `radial_cutoff`, and the network passes messages along these edges to predict the score of each
atom. Here, it has 4 layers of 64 hidden dimensions, the size of the pretrained model of Part 2, with a 5 Å radial
cutoff, and three options:

- `smooth_cutoff=True`: the messages vanish smoothly at the cutoff, so that the score does not jump when an atom
  crosses it.
- `normalize="smooth"`: the vector between two atoms at a distance r is divided by √(r² + a²), with a = 0.1 Å. It is a
  unit vector for r ≫ a, which keeps the direction between two close atoms at nearly full strength down to about
  0.1 Å, and vanishes smoothly when they overlap.
- `bond_direction_feature=True`: each edge also gets the orientation of its bond relative to the cell axes (1 along
  ⟨100⟩, 1/3 along ⟨111⟩), which the EGNN, equivariant under rotations, cannot see otherwise.

The loss is only on the coordinates (X): the atom types (A) and the lattice (L) are fixed here.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.loss.loss_parameters import (
    AtomTypeLossParameters, MSELossParameters)
from diffusion_for_multi_scale_molecular_dynamics.namespace import AXL
from diffusion_for_multi_scale_molecular_dynamics.score_network.cartesian_egnn_score_network import \
    CartesianEGNNScoreNetworkParameters

score_network_parameters = CartesianEGNNScoreNetworkParameters(
    num_atom_types=1, n_layers=4,
    coordinate_hidden_dimensions_size=64, coordinate_n_hidden_dimensions=4,
    message_hidden_dimensions_size=64, message_n_hidden_dimensions=4,
    node_hidden_dimensions_size=64, node_n_hidden_dimensions=4,
    radial_cutoff=5.0, smooth_cutoff=True, normalize="smooth", bond_direction_feature=True,
)

loss_parameters = AXL(A=AtomTypeLossParameters(lambda_weight=0.0),
                      X=MSELossParameters(lambda_weight=1.0),
                      L=MSELossParameters(lambda_weight=0.0))

## Target score matching

**Denoising score matching.** By default, the score network learns to denoise: the target is the opposite of the gaussian noise
added to each atom, normalized by σ. This is denoising score matching, based on the denoising score identity. Two noises compete: the added noise σ, and the noise due to thermal vibrations of the atoms. When the added noise is much smaller than the thermal noise, a single
target only points back to the clean configuration it came from which includes the thermal noise. This makes it hard for the score to describe the thermal spread of the data and the target is mostly random at low noise σ.

**The basin at 300 K.** The training configurations come from a molecular dynamics at 300 K: they are not a single
structure, but samples of the basin of configurations accessible at 300 K, distributed according to the Boltzmann
distribution p(x) ∝ exp(−βE(x)), with β = 1/(k<sub>B</sub>T). The score of this distribution is known exactly from the
forces: ∇log p(x) = βF(x).

**Target score matching.** Based on the target score identity, target score matching uses the forces instead: the
target is βF of the clean configuration. It directly models the energy basin through its forces, instead of relying
only on the sampled configurations. At low noise, this target is excellent since the noisy configurations are close
to the clean ones, and it is much better than denoising. At high noise, however, the forces of the clean configuration
say little about a noisy configuration far from it, and denoising is better.

**Combined score.** In the harmonic approximation, each atom vibrates around its equilibrium position with an
amplitude s that depends on the temperature: its thermal width. Its position then follows a Gaussian distribution of
variance s², which can be compared directly with the Gaussian added noise of variance σ². The noisy positions have a
variance s² + σ², of which a fraction κ = s²/(s² + σ²) comes from the thermal vibrations. This fraction is the mixer of
the combined score, the target used here:

target = κ σβF + (1 − κ) target<sub>DSM</sub>.

When σ ≪ s, κ ≈ 1 and the target is the forces; when σ ≫ s, κ ≈ 0 and it is the denoising target.

`compute_thermal_width_by_element` computes this thermal width automatically from a relaxed structure: the
interatomic force constants K give s² = k<sub>B</sub>T tr(K<sup>-1</sup>)/3. Here, the structure is the first
training frame, relaxed with Stillinger-Weber.

In [ ]:
from ase.calculators.lammpslib import LAMMPSlib
from ase.optimize import BFGS

from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.models.target_score_matching import \
    TargetScoreMatchingParameters
from diffusion_for_multi_scale_molecular_dynamics.oracle import SW_COEFFICIENTS_DIR
from diffusion_for_multi_scale_molecular_dynamics.utils.interatomic_force_constants import \
    compute_thermal_width_by_element

temperature = 300.0  # K, the temperature of the molecular dynamics that produced the training data

# Relax the first training frame with Stillinger-Weber, run in-process by LAMMPS.
relaxed_frame = frames[0].copy()
relaxed_frame.calc = LAMMPSlib(lmpcmds=["pair_style sw", f"pair_coeff * * {SW_COEFFICIENTS_DIR / 'Si.sw'} Si"],
                               log_file=None, keep_alive=True)
BFGS(relaxed_frame, logfile=None).run(fmax=1e-6)

thermal_width = compute_thermal_width_by_element(relaxed_frame, temperature)
print(f"Thermal width at {temperature} K: {thermal_width} Å")

# minimum_sigma and maximum_sigma only matter with bad frames (see Active training below).
target_score_matching_parameters = TargetScoreMatchingParameters(temperature=temperature,
                                                                 thermal_width=thermal_width,
                                                                 minimum_sigma=noise_parameters.sigma_min_cart,
                                                                 maximum_sigma=max(thermal_width.values()))

## Training

The `AXLDiffusionLightningModel` puts the score network, the loss and the target score matching together, with an
AdamW optimizer whose learning rate decreases by 10% when the validation loss stops improving for 4 epochs. A
Lightning `Trainer` runs the training, and stops it early if the validation loss does not improve for 25 epochs. It
writes:

- `models/version_0/checkpoints/epoch=N.ckpt`: the model after each epoch N.
- `training.log`: one line per epoch, with the training and validation losses.

Here, the training is limited to 5 epochs to keep the tutorial short; a real run should train on about 100,000 configurations for about 100 epochs depending on the complexity of the structure. Part 2 does not use this model, but a pretrained one: you can interrupt the kernel at any time during the training and go directly to Part 2. Jupyter then shows a `SystemExit` message, which can be ignored.

In [ ]:
import multiprocessing

from lightning import Trainer
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.callbacks.epoch_summary_callback import \
    EpochSummaryLogger
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.models.axl_diffusion_lightning_model import (
    AXLDiffusionLightningModel, AXLDiffusionParameters)
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.models.optimizer import OptimizerParameters
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.models.scheduler import \
    ReduceLROnPlateauSchedulerParameters

# The data loader workers must be started with fork instead of spawn to avoid crashing on Mac.
multiprocessing.set_start_method("fork", force=True)

diffusion_parameters = AXLDiffusionParameters(
    score_network_parameters=score_network_parameters,
    loss_parameters=loss_parameters,
    optimizer_parameters=OptimizerParameters(name="adamw", learning_rate=1.0e-4, weight_decay=5.0e-8),
    scheduler_parameters=ReduceLROnPlateauSchedulerParameters(factor=0.9, patience=4),
    elements=["Si"],
    target_score_matching_parameters=target_score_matching_parameters,
)
model = AXLDiffusionLightningModel(diffusion_parameters)

checkpoint_callback = ModelCheckpoint(monitor="validation_epoch_loss", mode="min",
                                      save_top_k=-1, every_n_epochs=1, filename="{epoch}")
early_stopping_callback = EarlyStopping(monitor="validation_epoch_loss", mode="min", patience=25)
epoch_summary_callback = EpochSummaryLogger(early_stopping_callback, output_path / "training.log")
trainer = Trainer(callbacks=[checkpoint_callback, early_stopping_callback,
                             LearningRateMonitor(logging_interval="epoch"), epoch_summary_callback],
                  logger=TensorBoardLogger(save_dir=str(output_path / "models"), name=""),
                  max_epochs=5,  # about 100 for a real run
                  log_every_n_steps=1)

try:
    trainer.fit(model, datamodule=data_module)
finally:
    data_module.clean_up()  # delete the working cache of the dataset, even if the training is interrupted

## Analysis

Once the training is complete, or interrupted, we can analyze it.

**Loss per epoch.** The training and validation losses at each epoch. Both should decrease as the model learns, then
flatten: they do not go to zero, since the noise added to each configuration is random and the network can only
predict its average. It is a red flag if the validation loss increases while the training loss keeps decreasing: the
model memorizes the training configurations instead of learning their distribution.

In [ ]:
import re

from matplotlib import pyplot as plt

# training.log has one line per epoch: "Epoch    3 | train_loss=  0.123456 | val_loss=  0.123456 | ...".
epochs, train_losses, validation_losses = [], [], []
for line in (output_path / "training.log").read_text().splitlines():
    match = re.match(r"Epoch\s+(\d+) \| train_loss=\s*(\S+) .*\| val_loss=\s*(\S+)", line)
    if match:
        epochs.append(int(match.group(1)))
        train_losses.append(float(match.group(2)))
        validation_losses.append(float(match.group(3)))

plt.plot(epochs, train_losses, "o-", label="Training")
plt.plot(epochs, validation_losses, "o-", label="Validation")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Active training

A trained model can still generate bad configurations, for example cells that are not diamond. The function
`active_train()` of `examples/03_train_diffusion_model.py` trains one epoch at a time. After each epoch, it generates
8 cells, and the bad ones, those in which not every atom has exactly 4 neighbors closer than 2.6 Å, are labelled with
Stillinger-Weber and appended to `bad_frames.traj`. The next epochs also train on these bad frames: they are not
noised, but given to the network at a low noise level, between `minimum_sigma` and `maximum_sigma`, and the network
learns their forces, which push the atoms back toward good configurations. The training stops when every generated
cell is good.

# Part 2: Generation

Part 2 uses the pretrained model `examples/references_files/pretrainedcartesianmodel_Si64_64-4_epoch43.ckpt`: a
Cartesian EGNN of 4 layers of 64 hidden dimensions, with a 7.5 Å radial cutoff, trained on Si64 without target score
matching (checkpoint of epoch 43). Since the EGNN only sees the neighbors of each atom within its cutoff, a model
trained on Si64 can generate Si8 cells. Part 2 does not use anything from Part 1: after the setup cells at the top,
it can be run alone.

This model was trained with a linear noise schedule, from 1.09×10<sup>-3</sup> to 2.18 Å, which is not the one of
Part 1: sampling must use the schedule of the training of the model.

## Sampling

The generator starts from random positions at σ<sub>max</sub>, and goes down the noise levels of the schedule to
σ<sub>min</sub> with a predictor-corrector algorithm. `total_time_steps` is the number of predictor steps, and
`number_of_corrector_steps` the number of corrector steps after each predictor step: with 1000 predictor steps and 2
corrector steps after each, the score network is evaluated 3000 times per sample.

**Predictor: noise reduction.** The predictor goes from the noise level σ<sub>i</sub> down to the next one,
σ<sub>i−1</sub>:

$$x \leftarrow x + (\sigma_i^2 - \sigma_{i-1}^2)\, S_\theta(x, \sigma_i) + \sqrt{\sigma_i^2 - \sigma_{i-1}^2}\, z,$$

where S<sub>θ</sub> is the score predicted by the network and z a standard Gaussian noise. It moves the atoms along the
score by the amount of noise removed between the two levels, plus a smaller random noise, so that the samples keep the
spread of the new level.

**Corrector: thermalization.** At the new noise level, the corrector runs a few steps of Langevin dynamics:

$$x \leftarrow x + \epsilon_i\, S_\theta(x, \sigma_{i-1}) + \sqrt{2\epsilon_i}\, z, \qquad
\epsilon_i = \frac{\epsilon}{2} \frac{\sigma_{i-1}^2}{\sigma_\text{min}^2}.$$

As with a thermostat in molecular dynamics, the score plays the role of the force, and z of the thermal kicks: the
corrector thermalizes the samples in the distribution of that noise level, which corrects the errors of the predictor.
ε is `corrector_step_epsilon`: with ε = 1.6 σ<sub>min</sub>², each random kick √(2ε<sub>i</sub>) is about
1.26 σ<sub>i−1</sub>.

Here, the cell is the cubic cell of the training data, and 16 cells are generated at once.

In [ ]:
import ase.io

from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.models.axl_diffusion_lightning_model import \
    AXLDiffusionLightningModel
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.noise_schedulers.noise_parameters import \
    NoiseParameters
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.generator.predictor_corrector_axl_generator import \
    PredictorCorrectorSamplingParameters

pretrained_model_path = TOP_DIR / "examples" / "references_files" / "pretrainedcartesianmodel_Si64_64-4_epoch43.ckpt"
pretrained_model = AXLDiffusionLightningModel.load_from_checkpoint(pretrained_model_path, map_location="cpu")
pretrained_model.eval()

# The noise schedule of the training of the pretrained model.
sigma_min = 1.088740e-3
pretrained_noise_parameters = NoiseParameters(total_time_steps=1000,  # number of predictor steps
                                              schedule_type="linear",
                                              sigma_min_cart=sigma_min,
                                              sigma_max_cart=2.177481,
                                              corrector_step_epsilon=1.6 * sigma_min**2)

training_frames = ase.io.read(TOP_DIR / "examples" / "references_files" / "si8_database.traj", index=":")
reference_frame = training_frames[0]
cell_dimensions = list(reference_frame.cell.lengths())

number_of_samples = 16
sampling_parameters = PredictorCorrectorSamplingParameters(num_atom_types=1,
                                                           number_of_atoms=len(reference_frame),
                                                           number_of_samples=number_of_samples,
                                                           use_fixed_lattice_parameters=True,
                                                           cell_dimensions=cell_dimensions,
                                                           number_of_corrector_steps=2)

## Generate from pure noise

The `LangevinGenerator` generates brand-new cells, without any constraint: here, 16 cells of 8 silicon atoms
(`number_of_samples`), all sampled together in a single batch. This should take about 2 minutes.

- `new_cells.traj`: the 16 generated cells.

In [ ]:
import torch

from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.data_module.utils import AXL_to_traj
from diffusion_for_multi_scale_molecular_dynamics.namespace import AXL, AXL_COMPOSITION
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.generator.langevin_generator import LangevinGenerator
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.sampling.diffusion_sampling import \
    create_batch_of_samples


def extract_structures(samples_batch):
    """Extract every sample of a create_batch_of_samples() batch as a list of ase.Atoms objects."""
    axl_batch = samples_batch[AXL_COMPOSITION]
    axl_list = [AXL(A=A.cpu().numpy(), X=X.cpu().numpy(), L=L.cpu().numpy())
                for A, X, L in zip(axl_batch.A, axl_batch.X, axl_batch.L)]
    return AXL_to_traj(axl_list, ["Si"])

# The number of generated samples is number_of_samples in sampling_parameters.
generator = LangevinGenerator(noise_parameters=pretrained_noise_parameters,
                              sampling_parameters=sampling_parameters,
                              axl_network=pretrained_model.axl_network)
with torch.no_grad():
    samples_batch = create_batch_of_samples(generator=generator,
                                            sampling_parameters=sampling_parameters,
                                            device=pretrained_model.device)

new_cells = extract_structures(samples_batch)
ase.io.write(output_path / "new_cells.traj", new_cells)

## Excise and repaint

To repaint, some atoms are kept fixed during sampling, and the model generates the other atoms around them. The
`ExciseAndRepaintSampleMaker` does it in a single call, `make_samples`, from a structure and an uncertainty per atom:

1. the atom selector picks the central atoms, here the `TopKAtomSelector` the most uncertain atom;
2. the `NearestNeighborsExcision` excises each central atom with its 2 closest neighbors;
3. these 3 atoms are placed at the center of the cell of the generated structures;
4. the `ConstrainedLangevinGenerator` keeps them fixed, and repaints the other 5 atoms around them.

This is the sample maker of the Generate stage of the active learning loop (see tutorial 03), where the uncertainty
per atom is the extrapolation grade of the MTP. Here, a random uncertainty picks a random atom of the first training
frame, and 16 cells are repainted around it. This should take about 2 minutes.

- `repainted_cells.traj`: the 16 repainted cells.

In [ ]:
import numpy as np

from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.data_module.utils import traj_to_AXL
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.atom_selector.top_k_atom_selector import (
    TopKAtomSelector, TopKAtomSelectorParameters)
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.excise_and_repaint_sample_maker import (
    ExciseAndRepaintSampleMaker, ExciseAndRepaintSampleMakerArguments)
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.excisor.nearest_neighbors_excisor import (
    NearestNeighborsExcision, NearestNeighborsExcisionArguments)

sample_maker_arguments = ExciseAndRepaintSampleMakerArguments(element_list=["Si"],
                                                              sample_box_size=cell_dimensions,
                                                              number_of_samples_per_substructure=number_of_samples)
sample_maker = ExciseAndRepaintSampleMaker(
    sample_maker_arguments=sample_maker_arguments,
    atom_selector=TopKAtomSelector(TopKAtomSelectorParameters(top_k_environment=1)),
    environment_excisor=NearestNeighborsExcision(NearestNeighborsExcisionArguments(number_of_neighbors=2)),
    noise_parameters=pretrained_noise_parameters,
    sampling_parameters=sampling_parameters,
    diffusion_model=pretrained_model.axl_network,
)

# A random uncertainty per atom: the most uncertain atom is excised with its 2 closest neighbors.
uncertainty_per_atom = np.random.rand(len(reference_frame))
repainted_structures, _, _ = sample_maker.make_samples(traj_to_AXL([reference_frame], ["Si"])[0], uncertainty_per_atom)

repainted_cells = AXL_to_traj(repainted_structures, ["Si"])
ase.io.write(output_path / "repainted_cells.traj", repainted_cells)

## Analysis

Once the cells are generated, we compare the new cells and the repainted cells with the training data, which gives
the expected values of good cells.

**Fraction of diamond cells.** The fraction of the cells in which every atom has exactly 4 neighbors closer than
2.6 Å, as in diamond silicon, whose first neighbors are at 2.35 Å and second neighbors at 3.84 Å. Every cell of the
training data is diamond.

In [ ]:
from ase.neighborlist import neighbor_list

bond_cutoff = 2.6  # Å, between the first and the second neighbors of diamond silicon
sets_of_cells = {"Training data": training_frames, "New cells": new_cells, "Repainted cells": repainted_cells}

for name, cells in sets_of_cells.items():
    is_diamond = [np.all(np.bincount(neighbor_list("i", atoms, bond_cutoff), minlength=len(atoms)) == 4)
                  for atoms in cells]
    print(f"{name}: {np.mean(is_diamond):.0%} of the cells are diamond")

**Bond angles.** The distribution of the angles between two bonds of the same atom. In diamond, every angle is the
tetrahedral angle, 109.47°; in the training data, the thermal vibrations spread them by about ±2.6°.

In [ ]:
def compute_bond_angles(cells):
    """Angles between every pair of bonds (closer than bond_cutoff) of every atom, in degrees."""
    angles = []
    for atoms in cells:
        atom_indices, bond_vectors = neighbor_list("iD", atoms, bond_cutoff)
        unit_bond_vectors = bond_vectors / np.linalg.norm(bond_vectors, axis=1, keepdims=True)
        for atom_index in range(len(atoms)):
            atom_bonds = unit_bond_vectors[atom_indices == atom_index]
            cosines = (atom_bonds @ atom_bonds.T)[np.triu_indices(len(atom_bonds), k=1)]
            angles.append(np.degrees(np.arccos(np.clip(cosines, -1.0, 1.0))))
    return np.concatenate(angles)


for name, cells in sets_of_cells.items():
    plt.hist(compute_bond_angles(cells), bins=np.arange(40.0, 182.0, 2.0), density=True, histtype="step", label=name)
plt.axvline(np.degrees(np.arccos(-1.0 / 3.0)), color="gray", linestyle="--", label="Tetrahedral angle")
plt.xlabel("Bond angle (°)")
plt.ylabel("Density")
plt.legend()
plt.show()

**Stillinger-Weber energy.** The Stillinger-Weber energy per atom of each cell, above the energy of the crystal at
equilibrium. The training data lies about 35 meV/atom above the crystal, from the thermal vibrations at 300 K. It is a
red flag if the generated cells are far above (hundreds of meV/atom): even if they look like diamond, some atoms are
at a wrong distance or angle. This figure would need more samples to better agree with training data, but only 16 were generated here to keep this tutorial lightweight. 

In [ ]:
from ase.build import bulk

from diffusion_for_multi_scale_molecular_dynamics.io.lammps.potential.stillinger_weber import \
    StillingerWeberPotential
from diffusion_for_multi_scale_molecular_dynamics.oracle import SW_COEFFICIENTS_DIR
from diffusion_for_multi_scale_molecular_dynamics.oracle.lammps_runner import InProcessLammpsRunner
from diffusion_for_multi_scale_molecular_dynamics.oracle.lammps_single_point_calculator import \
    LammpsSinglePointCalculator

potential = StillingerWeberPotential(sw_coefficients_file_path=SW_COEFFICIENTS_DIR / "Si.sw")
calculator = LammpsSinglePointCalculator(lammps_potential=potential, lammps_runner=InProcessLammpsRunner())


def compute_energies_per_atom(cells):
    """Stillinger-Weber energy per atom of every cell, in eV/atom."""
    return np.array([result.energy / len(result.atoms) for result in calculator.calculate_many(cells)])


crystal_energy_per_atom = compute_energies_per_atom([bulk("Si", "diamond", a=5.431, cubic=True)])[0]
for name, cells in sets_of_cells.items():
    energies_above_crystal = 1000.0 * (compute_energies_per_atom(cells) - crystal_energy_per_atom)
    print(f"{name}: {energies_above_crystal.mean():.0f} ± {energies_above_crystal.std():.0f} meV/atom above the crystal")
    plt.hist(energies_above_crystal, bins=30, density=True, histtype="step", label=name)
plt.xlabel("Energy above the crystal (meV/atom)")
plt.ylabel("Density")
plt.legend()
plt.show()